## **EXAMEN SEGUNDO PARCIAL 02/2026 - Inteligencia Artificial I (SIS420)**
### **Ejercicio 2 - Aprendizaje por Refuerzo**

**Nombre:** Plaza López Cesar Manuel

**Carrera:** Ingeniería de Sistemas

### **Nombre del juego:** Chomp 2x4 (El juego del chocolate envenenado)

El juego se desarrolla en un tablero de 2 filas por 4 columnas que simula una tableta de chocolate. La casilla superior izquierda $(0, 0)$ contiene veneno. Dos jugadores se turnan para "morder" el chocolate. En cada turno, un jugador elige una casilla que todavía esté disponible y retira del tablero esa pieza junto con todas las que se encuentren a su derecha y hacia abajo (formando un corte rectangular). Los turnos avanzan reduciendo progresivamente la tableta hasta que un jugador queda obligado a seleccionar la casilla envenenada $(0, 0)$, perdiendo la partida de inmediato (no hay empates).

In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm

class Board:
    def __init__(self, rows=2, cols=4):
        self.rows = rows
        self.cols = cols
        # 1 representa chocolate disponible, 0 representa mordido
        self.state = np.ones((self.rows, self.cols), dtype=int)

    def valid_moves(self):
        # Movimientos legales: casillas con chocolate excepto el veneno (0,0) si quedan otras opciones
        moves = [(r, c) for r in range(self.rows) for c in range(self.cols) if self.state[r, c] == 1]
        if len(moves) > 1 and (0, 0) in moves:
            moves.remove((0, 0))
        return moves

    def update(self, row, col):
        # Regla de Chomp: muerde (r, c) y todo lo que esté a su derecha y abajo
        if self.state[row, col] == 1:
            for r in range(row, self.rows):
                for c in range(col, self.cols):
                    self.state[r, c] = 0
        else:
            raise ValueError("¡Movimiento ilegal! Casilla ya mordida.")

    def is_game_over(self):
        # Si ya se comieron el veneno (0,0), el juego termina
        if self.state[0, 0] == 0:
            return True
        return False

    def reset(self):
        self.state = np.ones((self.rows, self.cols), dtype=int)

## **El Agente.-**

In [3]:
class Agent:
    def __init__(self, alpha=0.5, prob_exp=0.3):
        self.value_function = {}  # Tabla estado -> valor V(s)
        self.alpha = alpha        # Tasa de aprendizaje
        self.prob_exp = prob_exp  # Probabilidad de explorar (epsilon)
        self.positions = []       # Historial de estados en la partida

    def reset(self):
        self.positions = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()

        # Exploración: epsilon-greedy
        if explore and np.random.uniform(0, 1) < self.prob_exp:
            ix = np.random.choice(len(valid_moves))
            return valid_moves[ix]

        # Explotación: elegir la acción que deja al rival en el estado de menor valor
        min_rival_value = 1000
        best_row, best_col = valid_moves[0]

        for r, c in valid_moves:
            next_board = board.state.copy()
            # Simular mordida rectangular
            for i in range(r, board.rows):
                for j in range(c, board.cols):
                    next_board[i, j] = 0
            
            next_state = str(next_board.reshape(-1))
            val = self.value_function.get(next_state, 0.5)

            if val <= min_rival_value:
                min_rival_value = val
                best_row, best_col = r, c

        return best_row, best_col

    def update(self, board):
        # Guarda la fotografía del tablero
        self.positions.append(str(board.state.reshape(-1)))

    def reward(self, reward_val):
        # Actualización hacia atrás de la función de valor
        for p in reversed(self.positions):
            if self.value_function.get(p) is None:
                self.value_function[p] = 0.5
            self.value_function[p] += self.alpha * (reward_val - self.value_function[p])
            reward_val = self.value_function[p]

## **Gestor de Partidas (Game). -**

In [4]:
class Game:
    def __init__(self, player1, player2):
        self.players = [player1, player2]
        self.board = Board(rows=2, cols=4)

    def selfplay(self, rounds=5000):
        wins = [0, 0]
        for _ in tqdm(range(rounds), desc="Entrenando Agentes"):
            self.board.reset()
            for player in self.players:
                player.reset()

            current_player_idx = 0
            while not self.board.is_game_over():
                player = self.players[current_player_idx]
                r, c = player.move(self.board)
                self.board.update(r, c)
                player.update(self.board)

                # Si alguien muerde la última casilla segura, el otro queda forzado a perder
                if len(self.board.valid_moves()) == 0:
                    winner_idx = current_player_idx
                    loser_idx = 1 - current_player_idx
                    
                    self.players[winner_idx].reward(1.0)
                    self.players[loser_idx].reward(0.0)
                    wins[winner_idx] += 1
                    break

                current_player_idx = 1 - current_player_idx

        return wins

## **Entrenamiento (Self-Play).-**

In [5]:
agent1 = Agent(alpha=0.5, prob_exp=0.2)
agent2 = Agent(alpha=0.5, prob_exp=0.2)
game = Game(agent1, agent2)

victorias = game.selfplay(rounds=5000)
print(f"\nVictorias [Jugador 1, Jugador 2]: {victorias}")

Entrenando Agentes: 100%|██████████| 5000/5000 [00:09<00:00, 519.73it/s]


Victorias [Jugador 1, Jugador 2]: [1777, 3223]


## **Tabla de Valor y Guardado (pickle). -**

In [6]:
import pickle

# Ordenar la función de valor de mayor a menor probabilidad de victoria
funcion_de_valor = sorted(agent1.value_function.items(), key=lambda kv: kv[1], reverse=True)
tabla = pd.DataFrame({'estado': [x[0] for x in funcion_de_valor], 'valor_V(s)': [x[1] for x in funcion_de_valor]})

# Mostrar los primeros estados
print(f"Total de estados únicos descubiertos: {len(tabla)}")
display(tabla.head(10))

# Guardar el agente entrenado
with open('agente_chomp.pickle', 'wb') as handle:
    pickle.dump(agent1.value_function, handle, protocol=pickle.HIGHEST_PROTOCOL)

Total de estados únicos descubiertos: 14


,estado,valor_V(s)
0,[0 0 0 0 0 0 0 0],1.000000
1,[1 1 0 0 0 0 0 0],1.000000
2,[1 0 0 0 1 0 0 0],1.000000
3,[1 1 1 0 1 0 0 0],0.999939
4,[1 1 1 0 1 1 1 0],0.979485
5,[1 1 1 1 0 0 0 0],0.968689
6,[1 1 1 1 1 1 0 0],0.917408
7,[1 1 0 0 1 1 0 0],0.866211
8,[1 1 1 1 1 0 0 0],0.750000
9,[1 1 1 0 0 0 0 0],0.496094
